In [ ]:

import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.layers import Input, LSTM, RepeatVector, TimeDistributed, Dense

# Step 1: Load dataset
data = pd.read_csv('energy.csv')

# Step 2: Prepare dataset
if 'timestamp' in data.columns:
    data = data.drop('timestamp', axis=1)

data = data.select_dtypes(include=np.number)
data = data.fillna(data.mean())

# Step 3: Split dataset
split = int(len(data) * 0.8)
train = data.iloc[:split]
test = data.iloc[split-10:]

# Step 4: Normalize data
scaler = MinMaxScaler()
train = scaler.fit_transform(train)
test = scaler.transform(test)

# Step 5: Create time sequences
def create_sequences(data):
    X = []
    for i in range(len(data)-10):
        X.append(data[i:i+10])
    return np.array(X)

X_train = create_sequences(train)
X_test = create_sequences(test)

# Step 6: Create Autoencoder
model = tf.keras.Sequential([
    Input(shape=(10, X_train.shape[2])),
    LSTM(64),
    RepeatVector(10),
    LSTM(64, return_sequences=True),
    TimeDistributed(Dense(X_train.shape[2]))
])

# Step 7: Compile model
model.compile(optimizer='adam', loss='mse')

# Step 8: Train model
history = model.fit(
    X_train, X_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2
)

# Step 9: Predict and calculate error
train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

train_error = np.mean((X_train - train_pred)**2, axis=(1,2))
test_error = np.mean((X_test - test_pred)**2, axis=(1,2))

# Step 10: Detect anomalies
threshold = np.percentile(train_error, 95)
y_pred = (test_error > threshold).astype(int)

print("Threshold:", threshold)
print("Anomalies:", np.sum(y_pred))

plt.plot(history.history['loss'])
plt.title("Training Loss")
plt.show()
